# MOSAIC Postal Address Detector
**Detection-only.** Read-only against source data. Every finding requires steward review before any action.

| Cell | What it does | Procedure §§ |
|---|---|---|
| 1 - Overview | This document | — |
| 2 - Rule Reference | Full rule definitions, standardization options, decision workflow | All |
| 3 - Config | Parameters / widgets | — |
| 4 - Constants | Tags, severity, column taxonomy, US lookup tables, standardization rules | §1, §2 |
| 5 - Discovery | Inventory information_schema; identify address column groups per table | §1 |
| 6 - Profiler | Per-table, per-component stats: null rates, format distributions, case violations | §2, §4, §5 |
| 7 - AI Classifier | Confirm address groups; classify validation_status values; detect non-standard naming | §1, §5 |
| 8 - Rule Engine | One function per rule group; findings at address-group + column level | All |
| 9 - Write Results | Append findings to results Delta table | — |
| 10 - Summary | Blocking findings first, then by table and component | §5, Enforcement |

> **Hard constraint:** UPDATE, MERGE, DELETE, CTAS on source tables are never executed.
> **Key design:** Postal addresses span MULTIPLE columns. Detection is ADDRESS-GROUP-based:
>   Discovery first identifies which tables have an address group (address_1 + city + state + zip_code
>   or recognized aliases), then evaluates each group as a unit.
> **AI use:** ai_query() (1) confirms a table has a genuine address group vs. incidental city/state columns,
>   (2) detects non-Mosaic source naming that needs mapping, (3) classifies validation_status values.


# MOSAIC Postal Address Detector — Rule Reference

**Detection-only.** No source data is modified.

---

## §Data model (§1)
| Rule tag | What it detects | Required action |
|---|---|---|
| `MISSING_REQUIRED_COLUMN` | Address group present but missing one of: address_1, city, state, zip_code | Add required column per §1 minimum column set |
| `WRONG_COLUMN_NAME` | Non-Mosaic column name detected (address, x_address_2, addressline1, practiceaddressline1) | Map/rename to Mosaic convention in Silver ETL per §1 |
| `FULL_ADDRESS_ONLY` | Table has only a concatenated full_address column, no parsed components | Parse into address_1/city/state/zip_code per §1; do not store only concatenated string |
| `MISSING_VALIDATION_STATUS` | Address group present but no validation_status column | Add validation_status column per §1 recommendation |

## §Normalization (§2)
| Rule tag | What it detects | Required action |
|---|---|---|
| `ZIP_NOT_STRING` | zip_code stored as a numeric type (INT, DECIMAL) | Cast to STRING in Silver; preserve leading zeros |
| `ZIP_FORMAT_VIOLATION` | zip_code values not in ##### or #####-#### format for US | Reformat in Silver ETL; preserve ZIP+4 when source provides it |
| `STATE_FORMAT_VIOLATION` | state values not 2-char ANSI FIPS abbreviation | Map to 2-char abbreviation in Silver ETL |
| `UNIT_IN_ADDRESS_1` | APT/STE/FL/# unit designator patterns found in address_1 | Move unit to address_2 in Silver ETL per §2 |
| `PO_BOX_FORMAT_VIOLATION` | PO Box values not in standard "PO BOX {number}" format | Standardize in Silver ETL per §2 |
| `WHITESPACE_PUNCTUATION` | Leading/trailing whitespace or trailing commas/punctuation | Trim and clean in Silver ETL per §2 |
| `CASE_VIOLATION` | Wrong case for address components (UPPERCASE expected for USPS convention) | Apply UPPER() in Silver ETL per §2 |

## §Validation (§4, §5)
| Rule tag | What it detects | Required action |
|---|---|---|
| `NULL_REQUIRED_FIELD` | NULL or empty in a required component (address_1, city, state, zip_code) | Investigate source; set validation_status=unvalidated or quarantine |
| `HIGH_NULL_RATE` | > threshold% null in a required component | Escalate to steward; investigate upstream data loss |
| `INVALID_STATE_CODE` | state values not in the recognized US state/territory 2-char list | Map to valid codes or quarantine |
| `INVALID_ZIP_FORMAT` | zip_code values that fail US postal format check | NULL or quarantine per §5 |
| `PARTIAL_ADDRESS_UNVALIDATED` | Partial address (has city+state but no address_1) without validation_status=unvalidated | Set validation_status=unvalidated or document as intentionally partial per §4 |

## §Privacy (§7)
| Rule tag | What it detects | Required action |
|---|---|---|
| `PII_ADDRESS_RISK` | Address group detected — addresses linked to individuals are PII/PHI | Verify access controls, masking, and PHI handling |

## §Monitoring (§8)
| Rule tag | What it detects | Required action |
|---|---|---|
| `VALIDATION_STATUS_ABSENT` | Address group lacks any validation_status column | Add validation_status column; monitor unvalidated addresses per §8 |

---

## Enforcement
- Unstructured or unvalidated addresses in certified member mail/contact products are certification blockers.
- PHI address exposure in logs or unmasked lower environments escalates under security policies.


In [0]:
import re, json, uuid
from datetime import datetime
from typing import Any, Dict, List, Tuple, Set, Optional
from collections import defaultdict
from pyspark.sql import functions as F, DataFrame

def _w(name: str, default) -> str:
    try:
        dbutils.widgets.text(name, str(default))
        return dbutils.widgets.get(name)
    except Exception:
        return str(default)

CFG: Dict[str, Any] = {
    # Source
    "catalog":      _w("catalog",      "data_classification_source"),
    "schema":       _w("schema",       "postal_addresses_detector"),
    "table_filter": _w("table_filter", ""),
    "skip_views":   _w("skip_views",   "true").strip().lower() == "true",

    # Layer -- important: MISSING_VALIDATION_STATUS and PARTIAL_ADDRESS checks
    # are more strict in Silver/Gold than Bronze (Bronze may retain source-faithful data)
    "layer": _w("layer", "Unknown"),   # Bronze | Silver | Gold | Unknown

    # Default country for postal format validation
    "default_country": _w("default_country", "US"),

    # Sampling
    "sample_size": int(_w("sample_size", 10_000)),
    "seed":        int(_w("seed",        42)),

    # Detection thresholds
    "null_rate_threshold":   float(_w("null_rate_threshold",   "10.0")),
    "case_violation_threshold": float(_w("case_violation_threshold", "2.0")),
    "unit_in_addr1_threshold":  float(_w("unit_in_addr1_threshold",  "2.0")),

    # AI
    "ai_model":  _w("ai_model",  "databricks-meta-llama-3-3-70b-instruct"),
    "enable_ai": _w("enable_ai", "true").strip().lower() == "true",

    # Results
    "out_catalog": _w("out_catalog", "data_classification_target"),
    "out_schema":  _w("out_schema",  "data_classification_output"),
    "out_table":   _w("out_table",   "postal_findings"),
}

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().isoformat()

print(f"Run            : {RUN_ID}")
print(f"Scope          : {CFG['catalog']}.{CFG['schema']}")
print(f"Layer          : {CFG['layer']}")
print(f"Default country: {CFG['default_country']}")
print(f"AI             : {'on -> ' + CFG['ai_model'] if CFG['enable_ai'] else 'off (heuristic-only)'}")
print(f"Results        : {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")


In [0]:
# ---------------------------------------------------------------------------
# HOW TO ADD A NEW RULE:
#   1. Add tag to TAGS with procedure section reference
#   2. Add severity to SEVERITY
#   3. Add standardization options to STANDARDIZATION_RULES
#   4. Add detection logic in _check_address_group() in Cell 6
# ---------------------------------------------------------------------------

TAGS = {
    # §Data model
    "MISSING_REQUIRED_COLUMN":      "§Data-model",
    "WRONG_COLUMN_NAME":            "§Data-model",
    "FULL_ADDRESS_ONLY":            "§Data-model",
    "MISSING_VALIDATION_STATUS":    "§Data-model",
    # §Normalization
    "ZIP_NOT_STRING":               "§Normalization",
    "ZIP_FORMAT_VIOLATION":         "§Normalization",
    "STATE_FORMAT_VIOLATION":       "§Normalization",
    "UNIT_IN_ADDRESS_1":            "§Normalization",
    "PO_BOX_FORMAT_VIOLATION":      "§Normalization",
    "WHITESPACE_PUNCTUATION":       "§Normalization",
    "CASE_VIOLATION":               "§Normalization",
    # §Validation
    "NULL_REQUIRED_FIELD":          "§Validation",
    "HIGH_NULL_RATE":               "§Validation",
    "INVALID_STATE_CODE":           "§Validation",
    "INVALID_ZIP_FORMAT":           "§Validation",
    "PARTIAL_ADDRESS_UNVALIDATED":  "§Validation",
    # §Privacy
    "PII_ADDRESS_RISK":             "§Privacy",
    # §Monitoring
    "VALIDATION_STATUS_ABSENT":     "§Monitoring",
}

SEVERITY: Dict[str, int] = {
    "FULL_ADDRESS_ONLY":            10,
    "NULL_REQUIRED_FIELD":          10,
    "HIGH_NULL_RATE":                9,
    "INVALID_STATE_CODE":            8,
    "INVALID_ZIP_FORMAT":            8,
    "PARTIAL_ADDRESS_UNVALIDATED":   8,
    "ZIP_NOT_STRING":                8,
    "WRONG_COLUMN_NAME":             7,
    "MISSING_REQUIRED_COLUMN":       7,
    "ZIP_FORMAT_VIOLATION":          6,
    "STATE_FORMAT_VIOLATION":        6,
    "UNIT_IN_ADDRESS_1":             5,
    "PO_BOX_FORMAT_VIOLATION":       5,
    "CASE_VIOLATION":                5,
    "WHITESPACE_PUNCTUATION":        4,
    "PII_ADDRESS_RISK":              4,
    "MISSING_VALIDATION_STATUS":     3,
    "VALIDATION_STATUS_ABSENT":      3,
}

# ── Mosaic column taxonomy (§1) ───────────────────────────────────────────────
# MOSAIC_ADDR_COLS: canonical Mosaic Silver column names per §1
MOSAIC_ADDR_COLS = {
    "address_1", "address_2", "city", "state", "zip_code",
    "address_type", "validation_status", "country_code",
    "effective_from", "effective_to",
}
# Required minimum set (when an address group is present)
REQUIRED_ADDR_COLS = {"address_1", "city", "state", "zip_code"}

# ALIAS_MAP: non-standard column names → Mosaic canonical name
# Used to detect columns that need renaming in Silver ETL
ALIAS_MAP: Dict[str, str] = {
    # Tuva standard / common variants → address_1
    "address":              "address_1",
    "address1":             "address_1",
    "addressline1":         "address_1",
    "address_line1":        "address_1",
    "addr1":                "address_1",
    "addr_1":               "address_1",
    "streetaddress":        "address_1",
    "street_address":       "address_1",
    "street_address_1":     "address_1",
    "line1":                "address_1",
    "line_1":               "address_1",
    # Provider/NPI seed variants → address_1
    "practiceaddressline1": "address_1",
    "practice_address_line1":"address_1",
    "provideraddress1":     "address_1",
    # address_2 variants
    "x_address_2":          "address_2",
    "xaddress2":            "address_2",
    "address2":             "address_2",
    "addressline2":         "address_2",
    "address_line2":        "address_2",
    "addr2":                "address_2",
    "addr_2":               "address_2",
    "line2":                "address_2",
    "line_2":               "address_2",
    "practiceaddressline2": "address_2",
    "practice_address_line2":"address_2",
    "provideraddress2":     "address_2",
    # zip variants
    "zip":                  "zip_code",
    "zipcode":              "zip_code",
    "postal_code":          "zip_code",
    "postalcode":           "zip_code",
    "postcode":             "zip_code",
    # country variants
    "country":              "country_code",
    "country_cd":           "country_code",
    "iso_country":          "country_code",
}

# Full-address concatenation column names (§1: do not store only concatenated string)
FULL_ADDRESS_NAMES = {
    "full_address", "fulladdress", "complete_address", "mailing_address",
    "address_full", "formatted_address", "address_line", "raw_address",
    "address_string", "address_text",
}

# ── US State/Territory valid 2-char codes ─────────────────────────────────────
US_STATE_CODES = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN",
    "IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV",
    "NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN",
    "TX","UT","VT","VA","WA","WV","WI","WY",
    # Territories and DC
    "DC","PR","GU","VI","AS","MP","UM",
    # Military
    "AE","AP","AA",
}

# ── Unit designator patterns (should be in address_2, not address_1) ──────────
UNIT_PATTERN = re.compile(
    r"\b(APT|APARTMENT|SUITE|STE|UNIT|FL|FLOOR|ROOM|RM|BLDG|BUILDING|#)\s*[A-Z0-9]",
    re.I
)

# ── PO Box pattern ────────────────────────────────────────────────────────────
PO_BOX_PATTERN = re.compile(
    r"\b(P\.?O\.?\s*BOX|POST\s*OFFICE\s*BOX|PO\s*BOX)\b",
    re.I
)
# Canonical form: "PO BOX {number}"
PO_BOX_CANONICAL = re.compile(r"^PO BOX [0-9]+$")

# ── US ZIP code patterns ──────────────────────────────────────────────────────
ZIP_5_PATTERN      = re.compile(r"^[0-9]{5}$")
ZIP_PLUS4_PATTERN  = re.compile(r"^[0-9]{5}-[0-9]{4}$")
ZIP_VALID_PATTERN  = re.compile(r"^[0-9]{5}(-[0-9]{4})?$")

STANDARDIZATION_RULES: Dict[str, list] = {

    "MISSING_REQUIRED_COLUMN": [
        {"option_key": "add_required_column",
         "label":      "Add missing required address component column(s) in Silver",
         "sql":        "-- Add: address_1 STRING, city STRING, state STRING, zip_code STRING",
         "notes":      "Minimum column set per §1: address_1, city, state, zip_code. "
                       "Do not store only concatenated full_address without parsed components."},
    ],

    "WRONG_COLUMN_NAME": [
        {"option_key": "rename_in_silver",
         "label":      "Rename/alias non-standard column to Mosaic convention in Silver ETL",
         "sql":        "-- SILVER: col AS address_1  (or address_2 / zip_code per alias map)",
         "notes":      "Mosaic-specific naming per §1 and 2026-06-24 Architecture Sync: "
                       "address_1 (not address/addressline1), address_2 (not x_address_2/xaddress2), "
                       "zip_code (not zip/zipcode/postal_code). "
                       "Claims/provider: practiceaddressline1 → address_1, line2 → address_2."},
    ],

    "FULL_ADDRESS_ONLY": [
        {"option_key": "parse_into_components",
         "label":      "Parse concatenated address into address_1/city/state/zip_code components",
         "sql":        "-- Use address_parse_udf(raw_address, lit('US')) to decompose. "
                       "-- Quarantine rows where is_parsed=False.",
         "notes":      "Do not store only concatenated full_address in certified models without "
                       "parsed components in the same grain or linked conformed table (§1). "
                       "Use libpostal or equivalent on-cluster parser -- no cloud PHI export (§5)."},
    ],

    "MISSING_VALIDATION_STATUS": [
        {"option_key": "add_validation_status",
         "label":      "Add validation_status column to address group",
         "sql":        "-- ADD COLUMN validation_status STRING DEFAULT 'unvalidated'",
         "notes":      "validation_status recommended per §1. Monitor unvalidated addresses "
                       "in certified outreach (should be zero per §8)."},
    ],

    "ZIP_NOT_STRING": [
        {"option_key": "cast_to_string",
         "label":      "Cast zip_code to STRING in Silver to preserve leading zeros",
         "sql":        "-- SILVER: LPAD(CAST(zip_code AS STRING), 5, '0') AS zip_code",
         "notes":      "zip_code must be STRING, never numeric (§1). "
                       "Numeric storage silently drops leading zeros for states like NJ, CT, MA."},
    ],

    "ZIP_FORMAT_VIOLATION": [
        {"option_key": "reformat_zip",
         "label":      "Reformat zip_code to ##### or #####-#### in Silver",
         "sql":        "-- US: CASE WHEN zip_code RLIKE '^[0-9]{9}$' "
                       "THEN CONCAT(LEFT(zip_code,5),'-',RIGHT(zip_code,4)) "
                       "WHEN zip_code RLIKE '^[0-9]{5}$' THEN zip_code ELSE NULL END AS zip_code",
         "notes":      "Preserve ZIP+4 when source provides it; format as #####-#### (§1). "
                       "International postal codes preserved as-is (uppercase per §2)."},
    ],

    "STATE_FORMAT_VIOLATION": [
        {"option_key": "map_to_2char",
         "label":      "Map state to 2-character ANSI FIPS abbreviation in Silver",
         "sql":        "-- SILVER: UPPER(TRIM(state)) AS state  "
                       "-- then apply state name → code lookup",
         "notes":      "2-character abbreviation required for Tuva ANSI FIPS state tests (§1). "
                       "Map full names (California → CA) in Silver ETL."},
    ],

    "UNIT_IN_ADDRESS_1": [
        {"option_key": "move_unit_to_address_2",
         "label":      "Parse unit designator from address_1 into address_2 in Silver",
         "sql":        "-- Extract APT/STE/FL/# and suffix into address_2; "
                       "-- strip from address_1.",
         "notes":      "Put APT, STE, FL, # in address_2; avoid duplicating unit "
                       "in address_1 and address_2 (§2). "
                       "Use libpostal unit extraction or regex split on unit keywords."},
    ],

    "PO_BOX_FORMAT_VIOLATION": [
        {"option_key": "standardize_po_box",
         "label":      "Standardize PO Box to 'PO BOX {number}' format in Silver",
         "sql":        "-- SILVER: REGEXP_REPLACE(UPPER(TRIM(address_1)), "
                       "'P\\.?O\\.?\\s*BOX', 'PO BOX') AS address_1",
         "notes":      "Standardize as PO BOX {number} in address_1 (§2). "
                       "Do not split box number across address_1 and address_2 inconsistently."},
    ],

    "WHITESPACE_PUNCTUATION": [
        {"option_key": "trim_and_clean",
         "label":      "Trim and remove trailing punctuation in Silver transform",
         "sql":        "-- SILVER: TRIM(REGEXP_REPLACE(col, ',+$', '')) AS col",
         "notes":      "Trim whitespace; collapse internal double spaces; "
                       "remove trailing commas (§2)."},
        {"option_key": "trim_only",
         "label":      "Trim leading/trailing whitespace only",
         "sql":        "-- SILVER: TRIM(col) AS col",
         "notes":      "Minimum normalization per §2."},
    ],

    "CASE_VIOLATION": [
        {"option_key": "uppercase_usps",
         "label":      "Apply UPPERCASE to address components (USPS convention) in Silver",
         "sql":        "-- SILVER: UPPER(TRIM(col)) AS col",
         "notes":      "USPS convention: UPPERCASE for street/city/state/zip (§2). "
                       "Document casing choice per product in data dictionary. "
                       "Applies to US addresses; international casing documented separately."},
        {"option_key": "document_casing_exception",
         "label":      "Document casing convention in data dictionary",
         "sql":        "-- No transform. Document casing rule in data dictionary.",
         "notes":      "Proper case acceptable for display fields when steward-approved (§2). "
                       "Document per product before production load."},
    ],

    "NULL_REQUIRED_FIELD": [
        {"option_key": "investigate_source",
         "label":      "Investigate source for upstream data loss or missing mapping",
         "sql":        "-- No transform. Investigate source system and pipeline mapping.",
         "notes":      "Required field is NULL/empty. May indicate upstream data loss, "
                       "wrong join, or missing source mapping (§4). "
                       "Set validation_status=unvalidated for partial addresses."},
        {"option_key": "quarantine_null_address",
         "label":      "Route rows with NULL required field to quarantine",
         "sql":        "-- Route rows WHERE address_1 IS NULL OR city IS NULL "
                       "OR state IS NULL OR zip_code IS NULL to quarantine.",
         "notes":      "Invalid/partial addresses → quarantine per §5."},
    ],

    "HIGH_NULL_RATE": [
        {"option_key": "escalate_to_steward",
         "label":      "Escalate high null rate to steward for source investigation",
         "sql":        "-- No transform. Escalate to steward.",
         "notes":      "High null rate in required field may indicate systemic source issue. "
                       "Unstructured or unvalidated addresses in certified mail products "
                       "are certification blockers (§Enforcement)."},
    ],

    "INVALID_STATE_CODE": [
        {"option_key": "map_to_valid_code",
         "label":      "Map invalid state values to valid ANSI FIPS 2-char codes",
         "sql":        "-- Apply state name → 2-char code lookup in Silver. "
                       "-- Unmapped values → NULL or quarantine.",
         "notes":      "state must be 2-char ANSI FIPS abbreviation for Tuva alignment (§1). "
                       "Quarantine rows with unmappable state values (§5)."},
    ],

    "INVALID_ZIP_FORMAT": [
        {"option_key": "null_invalid_zip",
         "label":      "Set invalid zip_code values to NULL in Silver",
         "sql":        "-- SILVER: CASE WHEN zip_code RLIKE '^[0-9]{5}(-[0-9]{4})?$' "
                       "THEN zip_code ELSE NULL END AS zip_code",
         "notes":      "zip_code must match ##### or #####-#### for US (§1, §5). "
                       "Invalid values → NULL or quarantine; do not persist garbage ZIP strings."},
        {"option_key": "quarantine_invalid_zip",
         "label":      "Route rows with invalid zip_code to quarantine",
         "sql":        "-- Route rows WHERE NOT (zip_code RLIKE '^[0-9]{5}(-[0-9]{4})?$') "
                       "to quarantine table.",
         "notes":      "Use when steward needs to review invalid values before nulling."},
    ],

    "PARTIAL_ADDRESS_UNVALIDATED": [
        {"option_key": "set_validation_status_unvalidated",
         "label":      "Set validation_status=unvalidated for partial address rows",
         "sql":        "-- SILVER: CASE WHEN address_1 IS NULL "
                       "THEN 'unvalidated' ELSE validation_status END AS validation_status",
         "notes":      "Partial address allowed only when dictionary marks as intentionally partial "
                       "and validation_status=unvalidated is set (§4). "
                       "Do not fabricate address_1 from city/state to pass completeness checks (§4)."},
    ],

    "PII_ADDRESS_RISK": [
        {"option_key": "verify_pii_controls",
         "label":      "Verify PII/PHI access controls and masking policy",
         "sql":        "-- No transform. Document PHI controls in data dictionary.",
         "notes":      "Addresses linked to individuals are PII; patient addresses are PHI (§7). "
                       "Enforce PII masking and ETL per PHI storage policy. "
                       "Non-production: use masked/synthetic addresses unless break-glass approved."},
    ],

    "VALIDATION_STATUS_ABSENT": [
        {"option_key": "add_validation_status_column",
         "label":      "Add validation_status column to address group",
         "sql":        "-- ALTER TABLE ... ADD COLUMN validation_status STRING DEFAULT 'unvalidated'",
         "notes":      "Monitor unvalidated mail addresses in certified outreach "
                       "-- should be zero for certified mail lists (§8). "
                       "Required for DQ monitoring per §8."},
    ],
}

print(f"Constants loaded: {len(TAGS)} tags | {len(STANDARDIZATION_RULES)} standardization entries")
print(f"Required columns: {sorted(REQUIRED_ADDR_COLS)}")
print(f"Alias mappings  : {len(ALIAS_MAP)}")
print(f"US state codes  : {len(US_STATE_CODES)}")


In [0]:
# ---------------------------------------------------------------------------
# DISCOVERY -- key design:
# Unlike phone/email (single-column detection), postal addresses span MULTIPLE
# columns. Discovery identifies ADDRESS GROUPS per table:
#   A group = at least 2 of the required columns (address_1/aliases, city, state,
#   zip_code/aliases) present in the same table.
# Also detects: full_address-only tables, non-Mosaic column naming, zip_code type.
# ---------------------------------------------------------------------------

cat, sch = CFG["catalog"], CFG["schema"]

def _esc(name: str) -> str:
    return name.replace("`", "``")

view_clause = "AND table_type IN ('MANAGED', 'EXTERNAL')" if CFG["skip_views"] else ""
tables: List[str] = [
    r.table_name for r in spark.sql(f"""
        SELECT table_name FROM `{_esc(cat)}`.information_schema.tables
        WHERE  table_schema = '{_esc(sch)}' {view_clause}
        ORDER BY table_name
    """).collect()
]

if CFG["table_filter"].strip():
    pat = re.compile(CFG["table_filter"], re.I)
    tables = [t for t in tables if pat.search(t)]

if not tables:
    raise ValueError(f"No tables found in {cat}.{sch} -- check catalog/schema/table_filter.")

tbl_in = ", ".join(f"'{_esc(t)}'" for t in tables)

# Collect ALL columns with their types per table
table_col_info: Dict[str, List[Tuple[str, str]]] = defaultdict(list)  # tbl -> [(col, dtype)]
for r in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM   `{_esc(cat)}`.information_schema.columns
    WHERE  table_schema = '{_esc(sch)}' AND table_name IN ({tbl_in})
    ORDER BY table_name, ordinal_position
""").collect():
    table_col_info[r.table_name].append((r.column_name, r.data_type.upper()))


def _canonical(col: str) -> Optional[str]:
    """Map a column name to its Mosaic canonical name, or None if not address-related."""
    cl = col.lower().strip()
    if cl in {c.lower() for c in MOSAIC_ADDR_COLS}:
        return cl.replace("address_1", "address_1")  # already canonical
    return ALIAS_MAP.get(cl)


def _is_full_address_col(col: str) -> bool:
    return col.lower().strip() in FULL_ADDRESS_NAMES


# Build address groups per table
# An address group is a set of columns that together represent one address instance.
# For simplicity (and to handle tables with multiple address groups like
# mailing_address_1 + billing_address_1), we look for the base names and
# any prefixed variants (mailing_, billing_, home_, work_).
# The primary group uses unprefixed canonical names.

# Address component role detection
def _addr_role(col: str) -> Optional[str]:
    """Return the address role (address_1, city, state, zip_code, etc.) for a column."""
    cl = col.lower().strip()
    # Direct canonical match
    canonical_map = {c.lower(): c for c in MOSAIC_ADDR_COLS}
    if cl in canonical_map:
        return canonical_map[cl]
    # Alias match
    mapped = ALIAS_MAP.get(cl)
    if mapped:
        return mapped
    # Prefix variants: mailing_address_1, billing_city, home_zip_code etc.
    for prefix in ("mailing_", "billing_", "home_", "work_", "primary_", "secondary_",
                   "ship_", "physical_", "permanent_", "current_", "alt_"):
        if cl.startswith(prefix):
            base = cl[len(prefix):]
            if base in canonical_map:
                return canonical_map[base]
            mapped = ALIAS_MAP.get(base)
            if mapped:
                return mapped
    return None

# Per-table: group columns by address role
table_address_groups: Dict[str, Dict[str, dict]] = {}  # tbl -> {role -> {col, dtype, alias_of}}
table_full_address_cols: Dict[str, List[str]] = {}     # tbl -> [full_address col names]
table_zip_types: Dict[str, Dict[str, str]] = {}        # tbl -> {col_name -> dtype}

for tbl, cols in table_col_info.items():
    role_map = {}      # role -> (col_name, dtype, is_alias)
    full_addrs = []
    zip_types  = {}

    for col, dtype in cols:
        if _is_full_address_col(col):
            full_addrs.append(col)
            continue
        role = _addr_role(col)
        if role:
            is_alias = col.lower() not in {c.lower() for c in MOSAIC_ADDR_COLS}
            # Keep first match per role (primary group); prefix groups handled as separate
            if role not in role_map:
                role_map[role] = {"col": col, "dtype": dtype, "is_alias": is_alias}
        if col.lower() in ("zip_code", "zip", "zipcode", "postal_code", "postcode"):
            zip_types[col] = dtype

    # Qualify as an address group: need >= 2 of the 4 required roles present
    req_roles_present = sum(1 for r in REQUIRED_ADDR_COLS if r in role_map)
    if req_roles_present >= 2 or (full_addrs and req_roles_present == 0):
        if role_map:
            table_address_groups[tbl] = role_map
        if full_addrs:
            table_full_address_cols[tbl] = full_addrs
        if zip_types:
            table_zip_types[tbl] = zip_types

n_groups = len(table_address_groups) + len([t for t in table_full_address_cols if t not in table_address_groups])
print(f"Scope        : {cat}.{sch}")
print(f"Tables       : {len(tables)}")
print(f"Address tables: {n_groups}")
for tbl in sorted(set(list(table_address_groups) + list(table_full_address_cols))):
    roles = list(table_address_groups.get(tbl, {}).keys())
    fa    = table_full_address_cols.get(tbl, [])
    print(f"  {tbl}: roles={roles} full_addr_cols={fa}")


In [0]:
# ---------------------------------------------------------------------------
# PROFILER -- one SQL per table for ALL address component stats:
# Per-column: null_count, upper_count, whitespace_count, trailing_punct_count
# For address_1: unit_pattern_count, po_box_count, po_box_nonstandard_count
# For state: invalid_state_count, sample_invalids
# For zip_code: invalid_zip_count, zip_with_plus4_count
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")

def get_sample(tbl: str) -> DataFrame:
    fq = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    n  = CFG["sample_size"]
    try:
        return spark.sql(f"SELECT * FROM {fq} TABLESAMPLE ({n} ROWS)")
    except Exception:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fq}").collect()[0]["n"]
        return (
            spark.table(fq)
            .sample(withReplacement=False, fraction=min(1.0, n / max(1, total)), seed=CFG["seed"])
            .limit(n)
        )


def profile_address_group(tbl: str, role_map: dict) -> dict:
    """
    ONE SQL per table computing stats for ALL address component columns.
    Returns enriched role_map with per-column stats.
    """
    fq    = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    exprs = ["COUNT(*) AS __total__"]

    # Build valid state list for SQL IN clause
    state_list = ", ".join(f"'{s}'" for s in US_STATE_CODES)

    for role, info in role_map.items():
        col  = info["col"]
        cs   = f"`{_esc(col)}`"
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        v    = f"TRIM({cs})"

        exprs += [
            f"COUNT(*) - COUNT({cs}) AS `__null__{safe}`",
            # uppercase: has alpha chars and all alpha is uppercase
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND {v} = UPPER({v}) AND {v} RLIKE '[A-Za-z]' THEN 1 END) AS `__upper__{safe}`",
            # leading/trailing whitespace
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND {cs} != TRIM({cs}) "
            f"  THEN 1 END) AS `__ws__{safe}`",
            # trailing comma or punctuation
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) RLIKE ',+$' "
            f"  THEN 1 END) AS `__punct__{safe}`",
        ]

        if role == "address_1":
            exprs += [
                # unit designator in address_1 (APT/STE/FL/# etc.)
                f"COUNT(CASE WHEN {cs} IS NOT NULL AND UPPER(TRIM({cs})) RLIKE "
                f"  '\\b(APT|APARTMENT|SUITE|STE|UNIT|FL|FLOOR|ROOM|RM|BLDG|BUILDING|#)\\s*[A-Z0-9]' "
                f"  THEN 1 END) AS `__unit__{safe}`",
                # PO Box present
                f"COUNT(CASE WHEN {cs} IS NOT NULL AND UPPER(TRIM({cs})) RLIKE "
                f"  '\\b(P[.]?O[.]?\\s*BOX|POST\\s*OFFICE\\s*BOX)\\b' "
                f"  THEN 1 END) AS `__pobox__{safe}`",
                # PO Box but NOT in canonical form "PO BOX {digits}"
                f"COUNT(CASE WHEN {cs} IS NOT NULL AND UPPER(TRIM({cs})) RLIKE "
                f"  '\\b(P[.]?O[.]?\\s*BOX|POST\\s*OFFICE\\s*BOX)\\b' "
                f"  AND NOT (UPPER(TRIM({cs})) RLIKE '^PO BOX [0-9]+$') "
                f"  THEN 1 END) AS `__pobox_noncanon__{safe}`",
            ]

        if role == "state":
            exprs += [
                # state value not in valid 2-char US list
                f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
                f"  AND UPPER(TRIM({cs})) NOT IN ({state_list}) "
                f"  THEN 1 END) AS `__invalid_state__{safe}`",
            ]

        if role == "zip_code":
            exprs += [
                # valid US ZIP: 5 digits or 5+4 with hyphen
                f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
                f"  AND NOT (TRIM({cs}) RLIKE '^[0-9]{{5}}(-[0-9]{{4}})?$') "
                f"  THEN 1 END) AS `__invalid_zip__{safe}`",
                # ZIP+4 present (has the hyphen)
                f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) RLIKE "
                f"  '^[0-9]{{5}}-[0-9]{{4}}$' THEN 1 END) AS `__zip_plus4__{safe}`",
            ]

    try:
        row = spark.sql(f"SELECT {', '.join(exprs)} FROM {fq}").collect()[0].asDict()
    except Exception as e:
        print(f"  [WARN] Batch profile failed for {tbl}: {e}")
        return role_map

    total = row["__total__"] or 0
    for role, info in role_map.items():
        col  = info["col"]
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        info.update({
            "total":         total,
            "null_count":    row.get(f"__null__{safe}", 0) or 0,
            "upper_count":   row.get(f"__upper__{safe}", 0) or 0,
            "ws_count":      row.get(f"__ws__{safe}", 0) or 0,
            "punct_count":   row.get(f"__punct__{safe}", 0) or 0,
            "unit_count":    row.get(f"__unit__{safe}", 0) or 0,
            "pobox_count":   row.get(f"__pobox__{safe}", 0) or 0,
            "pobox_noncanon":row.get(f"__pobox_noncanon__{safe}", 0) or 0,
            "invalid_state": row.get(f"__invalid_state__{safe}", 0) or 0,
            "invalid_zip":   row.get(f"__invalid_zip__{safe}", 0) or 0,
            "zip_plus4":     row.get(f"__zip_plus4__{safe}", 0) or 0,
        })

    return role_map


def collect_samples(tbl: str, col: str, sample_df: DataFrame,
                    filter_expr=None, n: int = 5) -> list:
    cs = f"`{_esc(col)}`"
    try:
        df = sample_df.select(F.col(cs).alias("v")).filter(F.col("v").isNotNull())
        if filter_expr:
            df = df.filter(filter_expr)
        return [str(r["v"]) for r in df.limit(n).collect()]
    except Exception:
        return []


# Run profiler across all address tables
table_samples: Dict[str, DataFrame] = {}
profiled_groups: Dict[str, dict] = {}   # tbl -> enriched role_map

for tbl in sorted(set(list(table_address_groups) + list(table_full_address_cols))):
    sample_df = get_sample(tbl)
    table_samples[tbl] = sample_df

    role_map = table_address_groups.get(tbl, {})
    if role_map:
        role_map = profile_address_group(tbl, dict(role_map))

    # Collect sample values per component
    for role, info in role_map.items():
        col = info["col"]
        cs  = f"`{_esc(col)}`"
        info["sample_vals"] = collect_samples(tbl, col, sample_df)
        if role == "state" and info.get("invalid_state", 0) > 0:
            info["invalid_state_samples"] = collect_samples(
                tbl, col, sample_df,
                filter_expr=F.upper(F.trim(F.col(cs))).isin(list(US_STATE_CODES)) == False
            )
        if role == "zip_code" and info.get("invalid_zip", 0) > 0:
            info["invalid_zip_samples"] = collect_samples(
                tbl, col, sample_df,
                filter_expr=~F.col(cs).rlike(r"^[0-9]{5}(-[0-9]{4})?$")
            )

    profiled_groups[tbl] = role_map


print(f"Profiler done. {len(profiled_groups)} address table(s) profiled.")
for tbl, roles in profiled_groups.items():
    total = next(iter(roles.values()), {}).get("total", 0) if roles else 0
    print(f"  {tbl}: {len(roles)} role(s), {total:,} rows")


In [0]:
# ---------------------------------------------------------------------------
# AI CLASSIFIER -- chunked ai_query() calls (BATCH_SIZE tables per call).
#
# Three responsibilities:
# 1. ADDRESS GROUP CONFIRMATION: does this table genuinely store postal addresses
#    for individuals/entities, or are city/state columns incidental (e.g. a
#    lookup reference table)? Prevents false positives on state/city lookup tables.
# 2. VALIDATION STATUS ASSESSMENT: if a validation_status column exists, classify
#    its distinct values to detect unvalidated addresses in certified tiers.
# 3. NON-STANDARD NAMING: flag source column names that need Silver ETL remapping
#    (e.g. addressline1, practiceaddressline1, x_address_2).
# ---------------------------------------------------------------------------

BATCH_SIZE = 10   # tables per AI call (address context is richer than phone/email)

def _ai_query(prompt: str) -> str:
    safe = prompt.replace("'", "''")
    raw  = spark.sql(
        f"SELECT ai_query('{CFG['ai_model']}', '{safe}', failOnError => false) AS r"
    ).collect()[0]["r"]
    if hasattr(raw, "errorStatus") and raw.errorStatus:
        raise ValueError(raw.errorStatus)
    return raw.result if hasattr(raw, "result") else str(raw)

def _chunked(items: list, size: int = BATCH_SIZE):
    for i in range(0, len(items), size):
        yield items[i:i + size]


# Result dict: tbl -> {confirmed, entity_type, has_nonstandard_names, ai_error}
ai_results: Dict[str, dict] = {}

def classify_address_tables(tbls: List[str]) -> None:
    if not CFG["enable_ai"] or not tbls:
        for tbl in tbls:
            ai_results[tbl] = {
                "confirmed":           True,
                "entity_type":         "unknown",
                "has_nonstandard_names": any(
                    info.get("is_alias", False)
                    for info in profiled_groups.get(tbl, {}).values()
                ),
                "confidence":          "low",
            }
        return

    for chunk in _chunked(tbls):
        payload = json.dumps([{
            "table":   tbl,
            "columns": [
                {"col": info["col"], "role": role,
                 "is_alias": info.get("is_alias", False),
                 "samples": info.get("sample_vals", [])[:4]}
                for role, info in profiled_groups.get(tbl, {}).items()
            ] + [{"col": c, "role": "full_address", "samples": []}
                 for c in table_full_address_cols.get(tbl, [])],
        } for tbl in chunk], default=str)

        prompt = (
            "Healthcare data governance expert. Reply ONLY with a JSON array -- no prose, no markdown. "
            "For each table determine: "
            "(1) confirmed: true if this table stores postal addresses for patients, members, "
            "providers, or facilities. False if city/state columns are purely for reference/lookup "
            "(e.g. a US states reference table, a country code lookup). "
            "(2) entity_type: 'patient', 'member', 'provider', 'facility', 'vendor', 'other', 'reference_only'. "
            "(3) has_nonstandard_names: true if any column name should be renamed to Mosaic convention "
            "(address_1 not address/addressline1, address_2 not x_address_2, zip_code not zip/zipcode). "
            "(4) confidence: high/medium/low. "
            f"Tables: {payload}. "
            'Return: [{"table":"<t>","confirmed":true|false,"entity_type":"<e>",'
            '"has_nonstandard_names":true|false,"confidence":"high|medium|low"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                tbl = item.get("table", "")
                if tbl not in profiled_groups:
                    continue
                ai_results[tbl] = {
                    "confirmed":             item.get("confirmed", True),
                    "entity_type":           item.get("entity_type", "other"),
                    "has_nonstandard_names": item.get("has_nonstandard_names", False),
                    "confidence":            item.get("confidence", "low"),
                }
        except Exception as e:
            print(f"  [WARN] AI classification chunk failed: {e}")
            for tbl in chunk:
                ai_results[tbl] = {
                    "confirmed":             True,  # conservative: keep in scope
                    "entity_type":           "unknown",
                    "has_nonstandard_names": any(
                        info.get("is_alias", False)
                        for info in profiled_groups.get(tbl, {}).values()
                    ),
                    "confidence":            "low",
                    "ai_error":              str(e),
                }

    # Remove tables confirmed as reference-only
    for tbl, res in list(ai_results.items()):
        if not res.get("confirmed", True) and res.get("confidence") == "high":
            print(f"  [AI] {tbl} excluded: reference/lookup table (not an address table)")
            del profiled_groups[tbl]
            ai_results[tbl]["excluded"] = True


classify_address_tables(list(profiled_groups.keys()))

confirmed_tables = [t for t in profiled_groups if not ai_results.get(t, {}).get("excluded")]
print(f"AI classification done. {len(confirmed_tables)} confirmed address table(s).")
for tbl in confirmed_tables:
    res = ai_results.get(tbl, {})
    print(f"  {tbl}: entity={res.get('entity_type','?')} nonstandard_names={res.get('has_nonstandard_names','?')} conf={res.get('confidence','?')}")


In [0]:
# ---------------------------------------------------------------------------
# RULE ENGINE:
# One _check_address_group() function evaluates an entire address group.
# Findings are at two levels:
#   - Table level: MISSING_REQUIRED_COLUMN, WRONG_COLUMN_NAME, FULL_ADDRESS_ONLY,
#     VALIDATION_STATUS_ABSENT, PII_ADDRESS_RISK
#   - Column level: all normalization and validation findings
# column_name field uses the actual column name; for table-level findings
# it uses "-- address group --" as the column placeholder.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")


def _finding(tbl, col, role, tag, count, total, samples, action,
             hint=None, confidence="high",
             std_options=None, auto_decided_action=None,
             entity_type="unknown") -> dict:
    pct     = round(count / total * 100, 4) if total else 0.0
    options = std_options if std_options is not None else STANDARDIZATION_RULES.get(tag, [])
    decided = auto_decided_action if auto_decided_action is not None else None
    return {
        "run_id":                   RUN_ID,
        "run_ts":                   RUN_TS,
        "catalog":                  cat,
        "schema":                   sch,
        "table_name":               tbl,
        "column_name":              col,
        "address_role":             role,
        "entity_type":              entity_type,
        "data_type":                "STRING",
        "layer":                    CFG["layer"],
        "rule_ref":                 TAGS.get(tag, "§?"),
        "classification":           tag,
        "affected_count":           int(count),
        "affected_pct":             float(pct),
        "total_rows":               int(total),
        "sample_values":            json.dumps(samples, default=str),
        "recommended_action":       action,
        "dictionary_strategy_hint": hint,
        "standardization_rule":     json.dumps(options, ensure_ascii=False),
        "confidence":               confidence,
        "needs_steward_review":     decided is None,
        "decided_action":           decided,
        "decided_by":               None,
    }


def _check_address_group(tbl: str, role_map: dict) -> list:
    findings   = []
    ai_res     = ai_results.get(tbl, {})
    entity_type= ai_res.get("entity_type", "unknown")

    # Total rows from any available column
    total = next((info.get("total", 0) for info in role_map.values()), 0)

    # ── §Data model: FULL_ADDRESS_ONLY ────────────────────────────────────────
    fa_cols = table_full_address_cols.get(tbl, [])
    if fa_cols and not role_map:
        for fa_col in fa_cols:
            findings.append(_finding(tbl, fa_col, "full_address", "FULL_ADDRESS_ONLY",
                0, total, [],
                f"Table has concatenated address column '{fa_col}' but no parsed address "
                "components (address_1, city, state, zip_code). "
                "Do not store only a concatenated full_address in certified models without "
                "parsed components in the same grain or a linked conformed table (§1). "
                "Use libpostal or equivalent on-cluster parser -- no cloud PHI export (§5).",
                confidence="high",
                entity_type=entity_type,
            ))
        return findings   # no further checks without parsed components

    # ── §Data model: MISSING_REQUIRED_COLUMN ─────────────────────────────────
    missing_roles = REQUIRED_ADDR_COLS - set(role_map.keys())
    if missing_roles:
        findings.append(_finding(tbl, "-- address group --", "group", "MISSING_REQUIRED_COLUMN",
            0, total, [],
            f"Address group is missing required component(s): {sorted(missing_roles)}. "
            "Minimum column set per §1: address_1, city, state, zip_code. "
            "Add missing columns or confirm intentional partial address per §4.",
            confidence="high",
            entity_type=entity_type,
        ))

    # ── §Data model: WRONG_COLUMN_NAME ───────────────────────────────────────
    # Fire for each column that is an alias (not in Mosaic canonical naming)
    alias_cols = [(role, info) for role, info in role_map.items() if info.get("is_alias")]
    if alias_cols or ai_res.get("has_nonstandard_names"):
        for role, info in alias_cols:
            findings.append(_finding(tbl, info["col"], role, "WRONG_COLUMN_NAME",
                0, total, [],
                f"Column '{info['col']}' is not the Mosaic-standard name for {role}. "
                f"Mosaic convention (§1, 2026-06-24 Architecture Sync): "
                f"use '{role}' (not '{info['col']}'). "
                "Rename/alias in Silver ETL. "
                "address_1 not address/addressline1; address_2 not x_address_2/xaddress2; "
                "zip_code not zip/zipcode/postal_code.",
                confidence="high",
                auto_decided_action="rename_in_silver",
                entity_type=entity_type,
            ))

    # ── §Data model: VALIDATION_STATUS_ABSENT ────────────────────────────────
    has_val_status = "validation_status" in role_map
    if not has_val_status:
        findings.append(_finding(tbl, "-- address group --", "group", "VALIDATION_STATUS_ABSENT",
            0, total, [],
            "Address group has no validation_status column. "
            "Monitor unvalidated mail addresses in certified outreach per §8. "
            "Recommended: add validation_status STRING DEFAULT 'unvalidated'.",
            confidence="high",
            auto_decided_action="add_validation_status_column",
            entity_type=entity_type,
        ))

    # ── §Privacy: PII_ADDRESS_RISK ────────────────────────────────────────────
    findings.append(_finding(tbl, "-- address group --", "group", "PII_ADDRESS_RISK",
        total, total, [],
        f"Address group detected (entity_type: {entity_type}). "
        "Addresses linked to individuals are PII; patient addresses are PHI (§7). "
        "Enforce PII masking and ETL per PHI storage policy. "
        "Non-production: use masked/synthetic addresses unless break-glass approved (§7). "
        "Do not log full addresses in job logs or tickets (§7).",
        confidence="high",
        auto_decided_action="verify_pii_controls",
        entity_type=entity_type,
    ))

    # ── Per-column checks ─────────────────────────────────────────────────────
    for role, info in role_map.items():
        col        = info["col"]
        col_total  = info.get("total", total)
        null_count = info.get("null_count", 0)
        non_null   = col_total - null_count
        samples    = info.get("sample_vals", [])

        # §Normalization: ZIP_NOT_STRING
        if role == "zip_code":
            dtype = info.get("dtype", "STRING")
            if not dtype.startswith("STRING") and not dtype.startswith("VARCHAR") and not dtype.startswith("CHAR"):
                findings.append(_finding(tbl, col, role, "ZIP_NOT_STRING",
                    col_total, col_total, [],
                    f"zip_code column '{col}' has type {dtype} (not STRING). "
                    "zip_code must be STRING to preserve leading zeros for states like NJ, CT, MA (§1).",
                    confidence="high",
                    auto_decided_action="cast_to_string",
                    entity_type=entity_type,
                ))

        # §Validation: NULL_REQUIRED_FIELD
        if role in REQUIRED_ADDR_COLS and null_count > 0:
            null_pct = null_count / col_total * 100 if col_total else 0
            findings.append(_finding(tbl, col, role, "NULL_REQUIRED_FIELD",
                null_count, col_total, [],
                f"Required address component '{role}' (column: '{col}') has "
                f"{null_count:,} NULL value(s) ({null_pct:.1f}%). "
                "Required field is NULL/empty -- may indicate upstream data loss, "
                "wrong join, or missing source mapping (§4). "
                "Set validation_status=unvalidated for partial addresses.",
                confidence="high",
                entity_type=entity_type,
            ))
            # §Validation: HIGH_NULL_RATE
            if null_pct > CFG["null_rate_threshold"]:
                findings.append(_finding(tbl, col, role, "HIGH_NULL_RATE",
                    null_count, col_total, [],
                    f"{null_pct:.1f}% of rows have NULL '{role}'. "
                    "High null rate in required component -- escalate to steward (§4). "
                    "Unvalidated addresses in certified mail products are certification blockers.",
                    confidence="high",
                    entity_type=entity_type,
                ))

        # §Normalization: WHITESPACE_PUNCTUATION
        ws_count    = info.get("ws_count", 0) + info.get("punct_count", 0)
        if ws_count > 0:
            findings.append(_finding(tbl, col, role, "WHITESPACE_PUNCTUATION",
                ws_count, col_total, samples[:3],
                f"{ws_count:,} value(s) in '{col}' have leading/trailing whitespace "
                "or trailing punctuation/commas. Trim and clean in Silver ETL (§2).",
                confidence="high",
                auto_decided_action="trim_and_clean",
                entity_type=entity_type,
            ))

        # §Normalization: CASE_VIOLATION (UPPERCASE expected for USPS addresses)
        if non_null > 0 and role in ("address_1", "address_2", "city", "state"):
            upper_count = info.get("upper_count", 0)
            # Violation = non-null, non-empty, NOT already uppercase
            non_upper = max(0, non_null - upper_count)
            if non_upper > 0:
                viol_pct = non_upper / non_null * 100
                if viol_pct > CFG["case_violation_threshold"]:
                    findings.append(_finding(tbl, col, role, "CASE_VIOLATION",
                        non_upper, col_total, samples[:3],
                        f"{non_upper:,} value(s) ({viol_pct:.1f}%) in '{col}' are not UPPERCASE. "
                        "USPS convention requires UPPERCASE for street/city/state (§2). "
                        "Apply UPPER(TRIM(col)) in Silver ETL; document casing choice per product.",
                        confidence="medium",
                        entity_type=entity_type,
                    ))

        # §Normalization: UNIT_IN_ADDRESS_1
        if role == "address_1" and info.get("unit_count", 0) > 0:
            viol_pct = info["unit_count"] / non_null * 100 if non_null else 0
            if viol_pct > CFG["unit_in_addr1_threshold"]:
                findings.append(_finding(tbl, col, role, "UNIT_IN_ADDRESS_1",
                    info["unit_count"], col_total, samples[:3],
                    f"{info['unit_count']:,} value(s) in address_1 contain unit designators "
                    "(APT/STE/FL/#). Unit designators belong in address_2 (§2). "
                    "Parse and move unit to address_2 in Silver ETL.",
                    confidence="medium",
                    entity_type=entity_type,
                ))

        # §Normalization: PO_BOX_FORMAT_VIOLATION
        if role == "address_1" and info.get("pobox_noncanon", 0) > 0:
            findings.append(_finding(tbl, col, role, "PO_BOX_FORMAT_VIOLATION",
                info["pobox_noncanon"], col_total, samples[:3],
                f"{info['pobox_noncanon']:,} PO Box value(s) in address_1 are not in "
                "canonical 'PO BOX {number}' format. "
                "Standardize PO Box format in Silver ETL (§2).",
                confidence="high",
                auto_decided_action="standardize_po_box",
                entity_type=entity_type,
            ))

        # §Validation: INVALID_STATE_CODE
        if role == "state" and info.get("invalid_state", 0) > 0:
            inv_samples = info.get("invalid_state_samples", samples[:3])
            findings.append(_finding(tbl, col, role, "INVALID_STATE_CODE",
                info["invalid_state"], col_total, inv_samples,
                f"{info['invalid_state']:,} state value(s) are not valid 2-char ANSI FIPS "
                "abbreviations. Map to valid codes or quarantine (§1, §5).",
                confidence="high",
                entity_type=entity_type,
            ))

        # §Normalization: STATE_FORMAT_VIOLATION (state is non-null but > 2 chars)
        if role == "state" and non_null > 0:
            # Check via sample if state values are longer than 2 chars
            try:
                long_state = collect_samples(
                    tbl, col, table_samples.get(tbl, spark.range(0)),
                    filter_expr=F.length(F.trim(F.col(f"`{_esc(col)}`"))) > 2
                )
                if long_state:
                    # Count full-table long states
                    cs = f"`{_esc(col)}`"
                    long_count = spark.sql(
                        f"SELECT COUNT(*) AS n FROM `{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}` "
                        f"WHERE {cs} IS NOT NULL AND LENGTH(TRIM({cs})) > 2"
                    ).collect()[0]["n"] or 0
                    if long_count > 0:
                        findings.append(_finding(tbl, col, role, "STATE_FORMAT_VIOLATION",
                            long_count, col_total, long_state[:3],
                            f"{long_count:,} state value(s) are longer than 2 characters "
                            "(e.g. full state names). Map to 2-char ANSI FIPS abbreviation "
                            "in Silver ETL for Tuva alignment (§1).",
                            confidence="high",
                            entity_type=entity_type,
                        ))
            except Exception:
                pass

        # §Validation: INVALID_ZIP_FORMAT
        if role == "zip_code" and info.get("invalid_zip", 0) > 0:
            inv_zip_samples = info.get("invalid_zip_samples", samples[:3])
            findings.append(_finding(tbl, col, role, "INVALID_ZIP_FORMAT",
                info["invalid_zip"], col_total, inv_zip_samples,
                f"{info['invalid_zip']:,} zip_code value(s) do not match US format "
                "(##### or #####-####). NULL or quarantine invalid values (§5). "
                "International postal codes preserved as-is per §3.",
                confidence="high",
                entity_type=entity_type,
            ))

        # §Normalization: ZIP_FORMAT_VIOLATION for ZIP+4
        # Only informational if some ZIP+4 present but not all standardized
        if role == "zip_code" and non_null > 0 and info.get("zip_plus4", 0) > 0:
            # Just note presence -- not a violation, just confirm +4 is preserved
            pass  # ZIP+4 is good -- no finding needed

    # §Validation: PARTIAL_ADDRESS_UNVALIDATED
    if "address_1" in role_map and "city" in role_map:
        addr_info = role_map["address_1"]
        city_info = role_map["city"]
        addr_null = addr_info.get("null_count", 0)
        city_null = city_info.get("null_count", 0)
        # Rows where city is present but address_1 is NULL = partial address
        partial_count = max(0, addr_null - city_null)
        if partial_count > 0 and not has_val_status:
            findings.append(_finding(tbl, "-- address group --", "group",
                "PARTIAL_ADDRESS_UNVALIDATED",
                partial_count, total, [],
                f"Approximately {partial_count:,} row(s) appear to have city/state but "
                "no address_1 (partial address), and no validation_status column exists. "
                "Partial addresses allowed only when dictionary marks as intentionally partial "
                "and validation_status=unvalidated is set (§4). "
                "Do not fabricate address_1 from city/state to pass completeness checks.",
                confidence="medium",
                entity_type=entity_type,
            ))

    return findings


# ── Main loop ─────────────────────────────────────────────────────────────────
# _check_address_group() handles both cases:
#   - tables with a role_map (parsed component columns)
#   - full-address-only tables (role_map is empty, fa_cols check runs inside)
# The previous code had an extra call for the full-address-only case which
# caused every finding to be emitted twice. One call per table is correct.
all_findings: List[dict] = []

for tbl in confirmed_tables:
    role_map = profiled_groups.get(tbl, {})
    findings = _check_address_group(tbl, role_map)
    all_findings.extend(findings)
    print(f"  ok {tbl}: {len(findings)} finding(s)")

print(f"\nRule engine done. {len(all_findings)} total finding(s).")


In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, BooleanType)

SCHEMA = StructType([
    StructField("run_id",                   StringType(),  True),
    StructField("run_ts",                   StringType(),  True),
    StructField("catalog",                  StringType(),  True),
    StructField("schema",                   StringType(),  True),
    StructField("table_name",               StringType(),  True),
    StructField("column_name",              StringType(),  True),
    StructField("address_role",             StringType(),  True),
    StructField("entity_type",              StringType(),  True),
    StructField("data_type",                StringType(),  True),
    StructField("layer",                    StringType(),  True),
    StructField("rule_ref",                 StringType(),  True),
    StructField("classification",           StringType(),  True),
    StructField("affected_count",           LongType(),    True),
    StructField("affected_pct",             DoubleType(),  True),
    StructField("total_rows",               LongType(),    True),
    StructField("sample_values",            StringType(),  True),
    StructField("recommended_action",       StringType(),  True),
    StructField("dictionary_strategy_hint", StringType(),  True),
    StructField("standardization_rule",     StringType(),  True),
    StructField("confidence",               StringType(),  True),
    StructField("needs_steward_review",     BooleanType(), True),
    StructField("decided_action",           StringType(),  True),
    StructField("decided_by",               StringType(),  True),
])

out_fq  = f"`{CFG['out_catalog']}`.`{CFG['out_schema']}`.`{CFG['out_table']}`"
out_tbl = f"{CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}"

findings_df = spark.createDataFrame(all_findings or [], schema=SCHEMA)

if all_findings:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CFG['out_catalog']}`.`{CFG['out_schema']}`")
    findings_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(out_tbl)
    print(f"ok  {len(all_findings):,} finding(s) written to {out_fq}")
else:
    print("No findings generated.")

print(f"Run ID: {RUN_ID}")


In [0]:
BLOCKING = {
    "FULL_ADDRESS_ONLY", "NULL_REQUIRED_FIELD", "HIGH_NULL_RATE",
    "INVALID_STATE_CODE", "INVALID_ZIP_FORMAT", "WRONG_COLUMN_NAME",
    "MISSING_REQUIRED_COLUMN", "ZIP_NOT_STRING",
}

if not all_findings:
    print("No postal address findings. Run complete.")
else:
    fdf = findings_df

    # Section 1 -- Blocking (certification blockers)
    block_df = fdf.filter(F.col("classification").isin(BLOCKING)) \
                  .orderBy(F.col("affected_pct").desc())
    n_block = block_df.count()
    print("=" * 70)
    print(f"  SECTION 1 -- BLOCKING FINDINGS (certification blockers): {n_block}")
    print("=" * 70)
    if n_block:
        display(block_df.select(
            "table_name","column_name","address_role","entity_type",
            "classification","rule_ref","affected_count","affected_pct",
            "recommended_action","decided_action","decided_by"
        ))
    else:
        print("  None.")

    # Section 2 -- By classification
    print("\n" + "-" * 70)
    print("SECTION 2 -- Findings by classification")
    print("-" * 70)
    display(
        fdf.groupBy("classification","rule_ref","address_role")
           .agg(
               F.count("*").alias("findings"),
               F.countDistinct("table_name").alias("tables"),
               F.sum("affected_count").alias("total_affected"),
           ).orderBy(F.col("findings").desc())
    )

    # Section 3 -- By table
    print("\n" + "-" * 70)
    print("SECTION 3 -- Findings by table")
    print("-" * 70)
    display(
        fdf.groupBy("table_name","entity_type")
           .agg(
               F.count("*").alias("total_findings"),
               F.sum(F.when(F.col("classification").isin(BLOCKING), 1).otherwise(0)).alias("blocking"),
               F.collect_set("address_role").alias("roles_present"),
           ).orderBy(F.col("blocking").desc(), F.col("total_findings").desc())
    )

    # Section 4 -- Naming convention issues (§1)
    naming_df = fdf.filter(F.col("classification").isin(
        "WRONG_COLUMN_NAME","MISSING_REQUIRED_COLUMN","FULL_ADDRESS_ONLY","MISSING_VALIDATION_STATUS"
    ))
    n_naming = naming_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 4 -- Data model / naming issues ({n_naming})")
    print("  Mosaic convention: address_1, address_2, zip_code, validation_status")
    print("-" * 70)
    if n_naming:
        display(naming_df.select(
            "table_name","column_name","address_role","classification",
            "recommended_action","decided_action","decided_by"
        ).orderBy("classification","table_name"))

    # Section 5 -- Normalization issues
    norm_df = fdf.filter(F.col("classification").isin(
        "ZIP_FORMAT_VIOLATION","STATE_FORMAT_VIOLATION","UNIT_IN_ADDRESS_1",
        "PO_BOX_FORMAT_VIOLATION","WHITESPACE_PUNCTUATION","CASE_VIOLATION","ZIP_NOT_STRING"
    ))
    n_norm = norm_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 5 -- Normalization issues ({n_norm})")
    print("-" * 70)
    if n_norm:
        display(norm_df.select(
            "table_name","column_name","address_role","classification",
            "affected_count","affected_pct","sample_values",
            "recommended_action","decided_action","decided_by"
        ).orderBy("table_name","address_role","classification"))

    # Section 6 -- PII/PHI exposure
    pii_df = fdf.filter(F.col("classification") == "PII_ADDRESS_RISK")
    n_pii  = pii_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 6 -- PII/PHI address exposure ({n_pii})")
    print("  Patient addresses are PHI -- verify access controls and masking")
    print("-" * 70)
    if n_pii:
        display(pii_df.select(
            "table_name","entity_type","affected_count",
            "recommended_action","decided_action","decided_by"
        ).orderBy("table_name"))

    # Section 7 -- Engineer work queue
    work_df = fdf.filter(F.col("decided_action").isNotNull())
    n_work  = work_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 7 -- Engineer work queue ({n_work} decided)")
    print("-" * 70)
    if n_work:
        display(work_df.select(
            "table_name","column_name","address_role","classification",
            "decided_action","decided_by","standardization_rule"
        ).orderBy("table_name","address_role"))
    else:
        print("  No decisions recorded yet.")
        print(f"  Query: SELECT * FROM {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")
        print(f"  WHERE run_id = '{RUN_ID}' AND decided_action IS NULL")

    print("\n" + "=" * 70)
    print(f"  Run: {RUN_ID}")
    print(f"  Scope: {cat}.{sch}  |  Layer: {CFG['layer']}")
    print(f"  Address tables: {len(confirmed_tables)}")
    print(f"  Findings: {len(all_findings):,}  |  Blocking: {n_block}  |  Naming: {n_naming}  |  Normalization: {n_norm}  |  PII: {n_pii}")
    print("=" * 70)
    print("\nDetection-only. No source data was modified.")
    print("Every finding requires data steward review before any action.")
